In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [4]:
def make_coordinates(frame, line_parameters):
    if line_parameters[0] == None:
        x1,y1,x2,y2 = None
        return x1,y1,x2,y2
    slope, intercept = line_parameters
    y1 = frame.shape[0]
    y2 = int(y1*(3/5))
    x1 = int((y1-intercept)/slope)
    x2 = int((y2-intercept)/slope)
    return np.array([x1,y1,x2,y2])

In [5]:
def avg_slope_intercept(frame, lines):
    left_fit = []
    right_fit = []
    for line in lines:
        x1, y1, x2, y2 = line.reshape(4)
        parameters = np.polyfit((x1,x2), (y1,y2), 1)
        slope = parameters[0]
        intercept = parameters[1]
        if slope < 0:
            left_fit.append((slope, intercept))
        else:
            right_fit.append((slope, intercept))

    left_fit_avg = np.average(left_fit, axis = 0)
    right_fit_avg = np.average(right_fit, axis = 0)
    left_line = make_coordinates(frame, left_fit_avg)
    right_line = make_coordinates(frame, right_fit_avg)
    
    return np.array([left_line, right_line])

In [6]:
def display_line(frame, lines):
    line_image = np.zeros_like(frame)
    if lines is not None:
        for x1, y1, x2, y2 in lines:
            cv2.line(line_image, (x1, y1), (x2, y2), (0, 255, 0), 2, cv2.LINE_AA)
    return line_image

In [7]:
def roi(frame, mask_points):
    polygon = np.array([
        mask_points # 2 Dimentionsal Polygon
    ])
    mask = np.zeros_like(frame)
    cv2.fillPoly(mask, polygon, 255)
    masked_img = cv2.bitwise_and(frame, mask)
    return masked_img

In [8]:
def get_thresh(val):
    pass

In [9]:
def cannyEdge_detection(grey_frame, lower_th, higher_th):
    cannyEdge = cv2.Canny(grey_frame, lower_th, higher_th)
    return cannyEdge

In [10]:
def createMaskPoints(frame, mask_points_num):
    for i in range(0, mask_points_num):
        cv2.imshow('Points', frame)
        cv2.setMouseCallback('Points', point_position)
        cv2.waitKey(0)

In [11]:
def point_position(event, x, y, flag, params):
    if event == cv2.EVENT_LBUTTONDOWN:
        if len(mask_points) != 0:
            mask_points.insert(len(mask_points)-1, (x,y))
        else:
            mask_points.append((x,y))
    if event == cv2.EVENT_LBUTTONUP:
        cv2.destroyWindow('Points')

In [12]:
# Getting Frame and applying Gaussian blur after changing colour space to single channel
def get_frame():
    _, frame = cap.read()
    if frame is None:
        return None, None
    
    return frame, cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5,5), 0)

In [ ]:
cap = cv2.VideoCapture('road2.mp4')
mask_points_num = int(input('Pls enter the number of masking points'))
mask_points = []

while True:
    frame, grey_scale = get_frame()
    createMaskPoints(frame, mask_points_num)
    if len(mask_points) == mask_points_num:
        break
cv2.destroyAllWindows()

cv2.namedWindow('Thresholds')
cv2.createTrackbar('Lower_Threshold', 'Thresholds', 173, 255, get_thresh)
cv2.createTrackbar('Higher_Threshold', 'Thresholds', 255, 255, get_thresh)

while True:
    frame, grey_scale = get_frame()
    
    if frame is None:
        cv2.waitKey(0)
        print('End of the Video')
        break
        
    lower_th = cv2.getTrackbarPos('Lower_Threshold', 'Thresholds')
    higher_th = cv2.getTrackbarPos('Higher_Threshold', 'Thresholds')
    canny = cannyEdge_detection(grey_scale, lower_th, higher_th)
    
    masked_img = roi(canny, mask_points)

    lines = cv2.HoughLinesP(masked_img, 2, np.pi/180, 100, np.array([]), minLineLength=20, maxLineGap=10)
    
    avg_lines = avg_slope_intercept(frame, lines)
    
    line_img = display_line(frame, avg_lines)
    
    # combined_img = cv2.addWeighted(frame, .8, line_img, 1, 0)

    
   

    # cv2.imshow('video', frame)
    # cv2.imshow('canny', canny)
    # cv2.imshow('mask', masked_img)
    cv2.imshow('lines', line_img)
    # cv2.imshow('Fianl Work', combined_img)
    
    # plt.show()
    
    if cv2.waitKey(0) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

In [1]:
import cv2
import numpy as np

def make_coordinates(frame, line_parameters):
    if line_parameters[0] is None:
        return None, None, None, None
    slope, intercept = line_parameters
    y1 = frame.shape[0]
    y2 = int(y1 * (3 / 5))
    x1 = int((y1 - intercept) / slope)
    x2 = int((y2 - intercept) / slope)
    return np.array([x1, y1, x2, y2])

def avg_slope_intercept(frame, lines):
    left_fit = []
    right_fit = []
    for line in lines:
        x1, y1, x2, y2 = line.reshape(4)
        parameters = np.polyfit((x1, x2), (y1, y2), 1)
        slope = parameters[0]
        intercept = parameters[1]
        if slope < 0:
            left_fit.append((slope, intercept))
        else:
            right_fit.append((slope, intercept))

    if left_fit and right_fit:
        left_fit_avg = np.average(left_fit, axis=0)
        right_fit_avg = np.average(right_fit, axis=0)
        left_line = make_coordinates(frame, left_fit_avg)
        right_line = make_coordinates(frame, right_fit_avg)
        return np.array([left_line, right_line])
    return []

def display_line(frame, lines):
    line_image = np.zeros_like(frame)
    if lines is not None:
        for x1, y1, x2, y2 in lines:
            if x1 is not None and y1 is not None:
                cv2.line(line_image, (x1, y1), (x2, y2), (0, 255, 0), 2, cv2.LINE_AA)
    return line_image

def roi(frame, mask_points):
    polygon = np.array([mask_points])
    mask = np.zeros_like(frame)
    cv2.fillPoly(mask, polygon, 255)
    return cv2.bitwise_and(frame, mask)

def canny_edge_detection(grey_frame, lower_th, higher_th):
    return cv2.Canny(grey_frame, lower_th, higher_th)

def get_frame(cap):
    _, frame = cap.read()
    if frame is None:
        return None, None
    return frame, cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0)

# Get points for region of interest (ROI) using mouse click
def create_mask_points(frame, mask_points_num):
    mask_points = []
    def point_position(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            if len(mask_points) < mask_points_num:
                mask_points.append((x, y))
                cv2.circle(frame, (x, y), 10, (0, 255, 0), -1)
                cv2.putText(frame, "Points Remaining {}".format(mask_points_num-len(mask_points)), (10,10), cv2.FONT_HERSHEY_COMPLEX, 20, color=(150, 0,150), thickness=5)
                if len(mask_points) == mask_points_num:
                    cv2.destroyWindow('Points')
    cv2.imshow('Points', frame)
    cv2.setMouseCallback('Points', point_position)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    return mask_points

# Initialize video capture
cap = cv2.VideoCapture('road2.mp4')

# Set the number of mask points (region of interest)
mask_points_num = int(input('Please enter the number of masking points: '))
frame, _ = get_frame(cap)
mask_points = create_mask_points(frame, mask_points_num)

# Callback function for trackbar (Threshold adjustments)
def get_thresh(val):
    pass

# Create trackbars for adjusting the Canny edge detection thresholds
cv2.namedWindow('Thresholds')
cv2.createTrackbar('Lower_Threshold', 'Thresholds', 173, 255, get_thresh)
cv2.createTrackbar('Higher_Threshold', 'Thresholds', 255, 255, get_thresh)

# Process video frames
while True:
    frame, grey_scale = get_frame(cap)
    if frame is None:
        print('End of the Video')
        break
    
    # Get current threshold values from trackbars
    lower_th = cv2.getTrackbarPos('Lower_Threshold', 'Thresholds')
    higher_th = cv2.getTrackbarPos('Higher_Threshold', 'Thresholds')
        
    # Apply Canny edge detection
    canny = canny_edge_detection(grey_scale, lower_th, higher_th)
    cv2.imshow('edges', canny)
    
    # Mask the image to only focus on the region of interest
    masked_img = roi(canny, mask_points)
    
    # Perform Hough Line Transform to detect lines
    lines = cv2.HoughLinesP(masked_img, 2, np.pi / 180, 100, np.array([]), minLineLength=20, maxLineGap=10)
    # Calculate average line parameters and draw them
    if lines is not None:
        avg_lines = avg_slope_intercept(frame, lines)
        line_img = display_line(frame, avg_lines)
        combined_img = cv2.addWeighted(frame, 0.8, line_img, 1, 0)
        cv2.imshow('Final Work', combined_img)
    else:
        cv2.imshow('Final Work', frame)
    
    # Exit the loop when the user presses 'q'
    if cv2.waitKey(33) & 0xFF == ord('q'):
        break

# Cleanup
cv2.destroyAllWindows()


Please enter the number of masking points:  5


error: OpenCV(4.11.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\drawing.cpp:2426: error: (-215:Assertion failed) p.checkVector(2, CV_32S) > 0 in function 'cv::fillPoly'
